# Modeling 
Modeling in data science is the process of creating mathematical or algorithmic frameworks—such as linear regression, decision trees, or neural networks—to identify patterns, predict outcomes, or analyze relationships within data. It involves selecting algorithms, training them on data, and optimizing parameters to solve specific problems like forecasting, classification, or anomaly detection.

In [2]:
# import necessary libraries and dataset
import pandas as pd

df = pd.read_csv("../data/processed/amazon_sales_final.csv")

df.head()

,Unnamed: 0,OrderID,OrderDate,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,Quantity,...,Year,Month,Dayofweek,IsWeekEnd,NetRevenue,PricePerUnit,ShippingRatio,CustomerOrderCount,CustomerAvgSpend,CategoryAvgPrice
0,0,ORD0000001,2023-01-31,CUST001504,Vihaan Sharma,P00014,Drone Mini,Books,BrightLux,3,...,2023,1,1,False,319.77,106.62,0.00,3,872.77,302.02
1,1,ORD0000002,2023-12-30,CUST000178,Pooja Kumar,P00040,Microphone,Home & Kitchen,UrbanStyle,1,...,2023,12,5,True,238.80,259.64,0.01,2,435.86,302.76
2,2,ORD0000003,2022-05-10,CUST047516,Sneha Singh,P00044,Power Bank 20000mAh,Clothing,UrbanStyle,3,...,2022,5,1,False,94.58,36.02,0.05,2,248.94,305.20
3,3,ORD0000004,2023-07-18,CUST030059,Vihaan Reddy,P00041,Webcam Full HD,Home & Kitchen,Zenith,5,...,2023,7,1,False,142.72,31.93,0.03,6,1200.94,302.76
4,4,ORD0000005,2023-02-04,CUST048677,Aditya Kapoor,P00029,T-Shirt,Clothing,KiddoFun,2,...,2023,2,5,True,773.46,410.68,0.01,1,821.36,305.20


## Order value prediction

Target:
`TotalAmount`

Idea:
- Predict order value before it is finalized.

Strong predictors:
- Quantity
- UnitPrice
- Discount
- CategoryAvgPrice
- CustomerAvgSpend

Use case:
- price estimation systems
- checkout optimization
- dynamic pricing insights

In [2]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from prophet import Prophet
from pathlib import Path

st.title("Revenue Forecasting System")
st.caption("Multi-model time series forecasting & comparison")
st.divider()



# ── Load data ───────────────────────────────────────────────────────────────
DATA_PATH = "../data/processed/amazon_sales_final.csv"

@st.cache_data
def load():
    df = pd.read_csv(DATA_PATH)
    df["OrderDate"] = pd.to_datetime(df["OrderDate"], dayfirst=True, errors="coerce")
    df = df.dropna(subset=["OrderDate"])
    return df

df = load()

# ── Sidebar filters ─────────────────────────────────────────────────────────
st.sidebar.header("Global Filters")

date_range = st.sidebar.date_input(
    "Date Range",
    value=[df["OrderDate"].min(), df["OrderDate"].max()]
)
categories = st.sidebar.multiselect(
    "Categories", df["Category"].unique(), default=df["Category"].unique()
)
countries = st.sidebar.multiselect(
    "Countries", df["Country"].unique(), default=df["Country"].unique()
)

mask = (
    df["OrderDate"].between(
        pd.Timestamp(date_range[0]),
        pd.Timestamp(date_range[1])
    ) &
    df["Category"].isin(categories) &
    df["Country"].isin(countries)
)
fdf = df[mask]

# ─────────────────────────────────────────────────────────────
# LOAD DATA (expects fdf already filtered from Page 1 sidebar)
# ─────────────────────────────────────────────────────────────

df_ts = (
    fdf.set_index("OrderDate")
       .resample("ME")["TotalAmount"]
       .sum()
       .reset_index()
)

df_ts.columns = ["Month", "Revenue"]

# ─────────────────────────────────────────────────────────────
# FEATURE ENGINEERING (for ML models)
# ─────────────────────────────────────────────────────────────

for lag in range(1, 7):
    df_ts[f"lag_{lag}"] = df_ts["Revenue"].shift(lag)

df_ts["rolling_mean_3"] = df_ts["Revenue"].rolling(3).mean()
df_ts = df_ts.dropna()

# time features
df_ts["month_num"] = df_ts["Month"].dt.month
df_ts["year"] = df_ts["Month"].dt.year

# ─────────────────────────────────────────────────────────────
# TRAIN TEST SPLIT
# ─────────────────────────────────────────────────────────────

train = df_ts.iloc[:-6]
test = df_ts.iloc[-6:]

X_train = train.drop(["Month", "Revenue"], axis=1)
y_train = train["Revenue"]

X_test = test.drop(["Month", "Revenue"], axis=1)
y_test = test["Revenue"]

# ─────────────────────────────────────────────────────────────
# MODELS
# ─────────────────────────────────────────────────────────────

# 1. Naive baseline
naive_pred = test["lag_1"]

# 2. Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# 3. XGBoost
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

# ─────────────────────────────────────────────────────────────
# PROPHET MODEL
# ─────────────────────────────────────────────────────────────

prophet_df = df_ts[["Month", "Revenue"]].rename(columns={"Month": "ds", "Revenue": "y"})

prophet_model = Prophet()
prophet_model.fit(prophet_df)

future = prophet_model.make_future_dataframe(periods=6, freq="M")
forecast = prophet_model.predict(future)

prophet_pred = forecast["yhat"].iloc[-6:].values

# ─────────────────────────────────────────────────────────────
# METRICS FUNCTION
# ─────────────────────────────────────────────────────────────

def metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

results = pd.DataFrame({
    "Model": ["Naive", "Linear", "XGBoost", "Prophet"],
    "MAE": [
        metrics(y_test, naive_pred)["MAE"],
        metrics(y_test, lr_pred)["MAE"],
        metrics(y_test, xgb_pred)["MAE"],
        metrics(y_test, prophet_pred)["MAE"],
    ],
    "RMSE": [
        metrics(y_test, naive_pred)["RMSE"],
        metrics(y_test, lr_pred)["RMSE"],
        metrics(y_test, xgb_pred)["RMSE"],
        metrics(y_test, prophet_pred)["RMSE"],
    ],
    "R2": [
        metrics(y_test, naive_pred)["R2"],
        metrics(y_test, lr_pred)["R2"],
        metrics(y_test, xgb_pred)["R2"],
        metrics(y_test, prophet_pred)["R2"],
    ]
})

# ─────────────────────────────────────────────────────────────
# MODEL SELECTION
# ─────────────────────────────────────────────────────────────

model_choice = st.selectbox(
    "Select Model",
    ["Naive", "Linear Regression", "XGBoost", "Prophet"]
)

# ─────────────────────────────────────────────────────────────
# PREDICTION SELECTION
# ─────────────────────────────────────────────────────────────

if model_choice == "Naive":
    pred = naive_pred
elif model_choice == "Linear Regression":
    pred = lr_pred
elif model_choice == "XGBoost":
    pred = xgb_pred
else:
    pred = prophet_pred

# ─────────────────────────────────────────────────────────────
# VISUALIZATION
# ─────────────────────────────────────────────────────────────

chart_df = pd.DataFrame({
    "Month": test["Month"].values,
    "Actual": y_test.values,
    "Predicted": pred
})

fig = px.line(
    chart_df,
    x="Month",
    y=["Actual", "Predicted"],
    title=f"{model_choice} — Actual vs Predicted Revenue",
    template="plotly_white"
)

st.plotly_chart(fig, use_container_width=True)

# ─────────────────────────────────────────────────────────────
# METRICS TABLE
# ─────────────────────────────────────────────────────────────

st.subheader("Model Performance Comparison")
st.dataframe(results, use_container_width=True)

# ─────────────────────────────────────────────────────────────
# FUTURE FORECAST (Prophet-based)
# ─────────────────────────────────────────────────────────────

st.subheader("Future Forecast (Prophet)")

future_chart = forecast[["ds", "yhat"]].tail(12)

fig2 = px.line(
    future_chart,
    x="ds",
    y="yhat",
    title="Next Months Revenue Forecast",
    template="plotly_white"
)

st.plotly_chart(fig2, use_container_width=True)

2026-05-05 22:15:04.105 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-05 22:15:04.107 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-05 22:15:04.112 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-05 22:15:04.115 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-05 22:15:04.121 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-05 22:15:04.126 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-05 22:15:04.128 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-05 22:15:04.131 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

/tmp/ipykernel_23792/991470521.py:25: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["OrderDate"] = pd.to_datetime(df["OrderDate"], dayfirst=True, errors="coerce")
2026-05-05 22:15:06.441 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-05 22:15:06.443 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-05 22:15:06.444 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-05 22:15:06.451 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-05 22:15:06.453 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-05 22:15:06.455 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored wh

KeyboardInterrupt: 